# TB Screening — Colab workspace

Connects the three places the project lives:

- **GitHub** holds the code. It is cloned here, so this notebook always runs the latest pushed version.
- **Google Drive** holds the datasets and trained models, which are too large for Git.
- **Colab** supplies a GPU for training.

Run the cells from top to bottom. For training, first choose *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import tensorflow as tf

print("TensorFlow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "none - Runtime > Change runtime type > T4 GPU")

## 1. Connect Google Drive

Colab asks for permission to read your Drive the first time. Everything the project keeps in Drive sits under one folder.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/tb-screening-system"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive folder:", DRIVE_ROOT)
print("Contents:", sorted(os.listdir(DRIVE_ROOT)))

## 2. Get the code from GitHub

Clones the repository on the first run and pulls on later runs.

In [ ]:
REPO_URL = "https://github.com/derrickkimutai1/tb-screening-system.git"
WORK = "/content/tb-screening-system"

if os.path.isdir(os.path.join(WORK, ".git")):
    !git -C {WORK} pull --ff-only
else:
    !git clone {REPO_URL} {WORK}

%cd {WORK}
!git log --oneline -3

## 3. Bring the datasets over from Drive

The laptop writes the datasets to Drive as one archive, `datasets.zip`, using `scripts/backup_to_drive.py`. Copying one archive and unpacking it on Colab's own disk is much faster than reading 800 images one by one through the Drive connection.

In [ ]:
import shutil
import zipfile

archive = os.path.join(DRIVE_ROOT, "datasets.zip")
if not os.path.exists(archive):
    raise FileNotFoundError(
        "datasets.zip is not in Drive yet. Run scripts/backup_to_drive.py on the laptop."
    )

if not os.path.isdir("data/raw/shenzhen"):
    shutil.copy(archive, "/content/datasets.zip")
    with zipfile.ZipFile("/content/datasets.zip") as bundle:
        bundle.extractall(".")

print(len(os.listdir("data/raw/montgomery/MontgomerySet/CXR_png")), "Montgomery images")
print(len(os.listdir("data/raw/shenzhen/CXR_png")), "Shenzhen images")

## 4. Check the pipeline

Loads a batch from each split through `ml/preprocessing.py`, the same module the web application uses, so training here prepares images exactly as the app will.

In [ ]:
import csv
import sys

sys.path.insert(0, "ml")
from preprocessing import prepare_batch

with open("ml/artifacts/splits.csv", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

for split in ("train", "validation", "test"):
    subset = [r for r in rows if r["split"] == split][:8]
    batch = prepare_batch(r["filepath"] for r in subset)
    print(f"{split:<11} {batch.shape}  range [{batch.min():.2f}, {batch.max():.2f}]")

## 5. Save results back to Drive

Colab's disk is wiped when the session ends. Anything training writes to `ml/models` or `ml/artifacts` is copied to Drive here, where the laptop picks it up through Google Drive for desktop.

In [ ]:
for folder in ("ml/models", "ml/artifacts"):
    if os.path.isdir(folder):
        shutil.copytree(folder, os.path.join(DRIVE_ROOT, folder), dirs_exist_ok=True)
        print("saved", folder, "to Drive")